# Práctica de PySpark en Databricks

Notebook de práctica para acompañar las clases de PySpark. Está pensado para Databricks Free Edition, así que todo corre en serverless.

Son 10 módulos y un desafío al final. En cada módulo hay una explicación corta con un ejemplo y luego los ejercicios. Debajo de cada ejercicio hay una celda que comprueba el resultado: si imprime "Correcto", pasa al siguiente; si sale un AssertionError, lee el mensaje y revisa tu código.

Antes de empezar:
1. Conecta el notebook a Serverless (selector de cómputo, arriba a la derecha).
2. Ejecuta las celdas en orden con Shift + Enter, porque varios ejercicios usan resultados de los anteriores.
3. Vamos a usar la tabla `samples.nyctaxi.trips` (viajes de taxi en Nueva York), las tablas de `samples.tpch` y algunos DataFrames pequeños creados a mano.

Nota: en Free Edition no hay API de RDD ni `sparkContext` (serverless usa Spark Connect), así que trabajamos solo con DataFrames y SQL. La variable `spark` ya viene creada.

## 0. Preparación
Importamos lo que vamos a usar en todo el notebook y definimos dónde se guardan las tablas. En Free Edition ya existen el catálogo `workspace` y el esquema `default`, y tienes permiso de escritura ahí.

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

CATALOGO = "workspace"
ESQUEMA = "default"

def ok(mensaje="Correcto"):
    print(mensaje)

print("Versión de Spark:", spark.version)

Versión de Spark: 4.2.0


## Módulo 1. Crear DataFrames

Un DataFrame es básicamente una tabla distribuida: filas y columnas con nombre y tipo (eso es el esquema). La forma más simple de crear uno es desde una lista de Python con `spark.createDataFrame`. Los tipos los puede inferir Spark o los podemos declarar nosotros con `StructType`.

In [0]:
datos = [("Ana", "Ventas", 3200), ("Luis", "TI", 4100)]
df_ejemplo = spark.createDataFrame(datos, ["nombre", "area", "sueldo"])

df_ejemplo.show()
df_ejemplo.printSchema()   # tipos inferidos: string, string, long

+------+------+------+
|nombre|  area|sueldo|
+------+------+------+
|   Ana|Ventas|  3200|
|  Luis|    TI|  4100|
+------+------+------+

root
 |-- nombre: string (nullable = true)
 |-- area: string (nullable = true)
 |-- sueldo: long (nullable = true)



### Ejercicio 1.1: DataFrame con esquema explícito
Crea el DataFrame `empleados` con los datos de abajo usando un esquema explícito con estos campos:

| campo | tipo |
|---|---|
| `nombre` | `StringType()` |
| `area` | `StringType()` |
| `sueldo` | `IntegerType()` |
| `fecha_ingreso` | `DateType()` |

In [0]:
datos_empleados = [
    ("Ana",   "Ventas",   3200, date(2019, 3, 1)),
    ("Luis",  "TI",       4100, date(2020, 7, 15)),
    ("Marta", "TI",       4800, date(2017, 1, 10)),
    ("Jorge", "Ventas",   2900, date(2022, 9, 5)),
    ("Sofía", "Finanzas", 3900, date(2018, 11, 20)),
    ("Pedro", "Finanzas", 3500, date(2021, 5, 3)),
]

esquema_empleados = StructType([
    StructField("nombre", StringType(), True),
    StructField("area", StringType(), True),
    StructField("sueldo", IntegerType(), True),
    StructField("fecha_ingreso", DateType(), True),
])

empleados = spark.createDataFrame(datos_empleados, esquema_empleados)

empleados.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- area: string (nullable = true)
 |-- sueldo: integer (nullable = true)
 |-- fecha_ingreso: date (nullable = true)



In [0]:
# comprobación
assert empleados.count() == 6, "Deberías tener 6 filas"
assert dict(empleados.dtypes) == {"nombre": "string", "area": "string", "sueldo": "int", "fecha_ingreso": "date"}, \
    f"Revisa los tipos: {empleados.dtypes}"
ok()

Correcto


### Ejercicio 1.2: filas y columnas
Guarda en `n_filas` el número de filas de `empleados` y en `n_columnas` el número de columnas.

Pista: `count()` y el atributo `.columns`.

In [0]:
n_filas = empleados.count()
n_columnas = len(empleados.columns)

print(n_filas, n_columnas)

6 4


In [0]:
# comprobación
assert n_filas == 6 and n_columnas == 4, "Esperaba 6 filas y 4 columnas"
ok()

Correcto


## Módulo 2. Explorar una tabla real

Con `spark.read.table("catalogo.esquema.tabla")` leemos una tabla de Unity Catalog. Vamos a usar `samples.nyctaxi.trips`, que ya viene en el workspace.

`display()` muestra una tabla interactiva (desde ahí también se pueden hacer gráficos) y `show()` imprime texto plano.

In [0]:
trips = spark.read.table("samples.nyctaxi.trips")
display(trips.limit(5))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022


In [0]:
trips.limit(5).show()

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+



### Ejercicio 2.1: esquema y columnas
Imprime el esquema de `trips` y guarda la lista con los nombres de las columnas en `columnas`.

In [0]:
trips.printSchema()
columnas = trips.columns

print(columnas)

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)

['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'fare_amount', 'pickup_zip', 'dropoff_zip']


In [0]:
# comprobación
assert isinstance(columnas, list) and "fare_amount" in columnas and "pickup_zip" in columnas
ok()

Correcto


### Ejercicio 2.2: cantidad de viajes
Guarda en `n_viajes` el número total de filas de `trips`.

In [0]:
n_viajes = trips.count()
print(f"Hay {n_viajes:,} viajes")

Hay 21,932 viajes


In [0]:
# comprobación
assert isinstance(n_viajes, int) and n_viajes > 0
ok()

Correcto


### Ejercicio 2.3: estadísticas descriptivas
Crea un DataFrame `estadisticas` con el conteo, media, desviación estándar, mínimo y máximo de `trip_distance` y `fare_amount`.

Pista: hay un método de DataFrame que hace justo esto, parecido al `.describe()` de pandas.

In [0]:
estadisticas = trips.describe("trip_distance", "fare_amount")
display(estadisticas)

summary,trip_distance,fare_amount
count,21932,21932
mean,2.852829199343427,12.34872697428415
stddev,3.4399536210623523,10.327887859233329
min,0.0,-8.0
max,30.6,275.0


In [0]:
# comprobación
assert estadisticas.columns == ["summary", "trip_distance", "fare_amount"]
ok()

Correcto


### Ejercicio 2.4: rango de fechas
¿Entre qué fechas están los viajes? Crea `rango` con dos columnas: `desde` (el mínimo de `tpep_pickup_datetime`) y `hasta` (el máximo).

In [0]:
rango = trips.select(
    F.min("tpep_pickup_datetime").alias("desde"),
    F.max("tpep_pickup_datetime").alias("hasta")
)
display(rango)

desde,hasta
2016-01-01T00:04:30.000Z,2016-02-29T23:51:20.000Z


In [0]:
# comprobación
fila = rango.first()
assert rango.columns == ["desde", "hasta"] and fila["desde"] <= fila["hasta"]
ok()

Correcto


## Módulo 3. Seleccionar y filtrar

Lo básico que vamos a usar:

| Para | Se usa |
|---|---|
| elegir columnas | `df.select("a", "b")` |
| filtrar filas | `df.filter(F.col("a") > 10)` (o `where`) |
| combinar condiciones | `&` (y), `\|` (o), `~` (no), cada condición entre paréntesis |
| valor dentro de una lista | `F.col("a").isin(1, 2, 3)` |
| ordenar | `df.orderBy(F.desc("a"))` |
| quitar duplicados | `df.distinct()` |

Ojo: entre columnas no se usa `and` / `or` de Python, se usa `&` y `|`.

In [0]:
# Ejemplo: viajes de más de 5 millas, ordenados por distancia
display(
    trips.where(F.col("trip_distance") > 5)
         .select("tpep_pickup_datetime", "trip_distance", "fare_amount")
         .orderBy(F.desc("trip_distance"))
         .limit(5)
)

tpep_pickup_datetime,trip_distance,fare_amount
2016-02-22T21:17:27.000Z,30.6,95.0
2016-01-07T04:07:58.000Z,26.3,0.0
2016-02-06T12:18:51.000Z,25.8,71.0
2016-02-16T22:30:58.000Z,25.55,73.5
2016-02-17T22:23:14.000Z,25.46,130.0


### Ejercicio 3.1: seleccionar columnas
Crea `viajes_basicos` solo con estas columnas y en este orden: `tpep_pickup_datetime`, `trip_distance`, `fare_amount`, `pickup_zip`.

In [0]:
viajes_basicos = trips.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "pickup_zip"
)
display(viajes_basicos.limit(5))

tpep_pickup_datetime,trip_distance,fare_amount,pickup_zip
2016-02-13T21:47:53.000Z,1.4,8.0,10103
2016-02-13T18:29:09.000Z,1.31,7.5,10023
2016-02-06T19:40:58.000Z,1.8,9.5,10001
2016-02-12T19:06:43.000Z,2.3,11.5,10044
2016-02-23T10:27:56.000Z,2.6,18.5,10199


In [0]:
# comprobación
assert viajes_basicos.columns == ["tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip"]
ok()

Correcto


### Ejercicio 3.2: dos condiciones
Crea `viajes_largos` con los viajes de más de 10 millas y con tarifa menor a 100.

In [0]:
viajes_largos = trips.filter(
    (F.col("trip_distance") > 10) & (F.col("fare_amount") < 100)
)
print(viajes_largos.count())

1120


In [0]:
# comprobación
assert viajes_largos.filter((F.col("trip_distance") <= 10) | (F.col("fare_amount") >= 100)).count() == 0, \
    "Hay filas que no cumplen ambas condiciones"
assert viajes_largos.count() == trips.filter("trip_distance > 10 AND fare_amount < 100").count()
ok()

Correcto


### Ejercicio 3.3: los 5 más caros
Crea `top5_caros` con los 5 viajes de mayor `fare_amount`, ordenados de mayor a menor.

In [0]:
top5_caros = trips.orderBy(F.desc("fare_amount")).limit(5)
display(top5_caros)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-12T20:55:19.000Z,2016-02-12T21:52:38.000Z,20.85,275.0,10013,7008
2016-02-29T12:16:16.000Z,2016-02-29T12:16:53.000Z,0.0,260.0,8876,8876
2016-01-30T22:28:42.000Z,2016-01-30T22:30:14.000Z,0.0,188.0,7974,7974
2016-02-17T22:23:14.000Z,2016-02-17T23:06:15.000Z,25.46,130.0,10103,7901
2016-01-28T17:36:17.000Z,2016-01-28T18:27:48.000Z,21.3,115.0,10017,11042


In [0]:
# comprobación
maximo = trips.agg(F.max("fare_amount")).first()[0]
filas = top5_caros.collect()   # son 5 filas, el collect no es problema
assert len(filas) == 5 and filas[0]["fare_amount"] == maximo
assert all(filas[i]["fare_amount"] >= filas[i + 1]["fare_amount"] for i in range(4)), "No está ordenado de mayor a menor"
ok()

Correcto


### Ejercicio 3.4: valores distintos
¿Cuántos códigos postales de origen (`pickup_zip`) distintos hay? Guarda el número en `n_zips`.

In [0]:
n_zips = trips.select("pickup_zip").dropDuplicates().count()
print(n_zips)

128


In [0]:
# comprobación
assert n_zips == trips.groupBy("pickup_zip").count().count()
ok()

Correcto


### Ejercicio 3.5: filtrar por lista
Crea `desde_midtown` con los viajes cuyo `pickup_zip` sea 10001, 10011 o 10019. Usa `isin`.

In [0]:
%sql
SELECT
    pickup_zip,
    COUNT(1) AS count
FROM samples.nyctaxi.trips
WHERE pickup_zip IN (10001, 10011, 10019)
GROUP BY pickup_zip


pickup_zip,count
10001,1227
10011,1129
10019,504


In [0]:
desde_midtown = trips.where(
    F.col("pickup_zip").isin([10001, 10011, 10019])
)
display(desde_midtown.groupBy("pickup_zip").count())

pickup_zip,count
10001,1227
10011,1129
10019,504


In [0]:
# comprobación
zips = {r["pickup_zip"] for r in desde_midtown.select("pickup_zip").distinct().collect()}
assert zips <= {10001, 10011, 10019} and len(zips) > 0
ok()

Correcto


## Módulo 4. Columnas nuevas, fechas y condiciones

`withColumn("nombre", expresion)` agrega una columna nueva (o reemplaza una que ya existe). Algunas expresiones que vamos a necesitar:

- Aritmética: `F.col("a") * 2`, `F.round(expr, 2)`
- Fechas y horas: `F.hour`, `F.dayofweek` (1 = domingo, 7 = sábado), `F.to_date`, `F.unix_timestamp` (en segundos)
- Condiciones: `F.when(cond, valor).when(...).otherwise(valor)`
- División segura: `F.try_divide(a, b)`, que devuelve `null` cuando `b` es 0

En serverless está activo el modo ANSI, así que dividir entre cero o hacer un cast inválido lanza error en lugar de devolver `null`. Por eso usamos `try_divide`.

Recuerda que los DataFrames son inmutables: cada `withColumn` devuelve un DataFrame nuevo, hay que guardarlo en una variable.

### Ejercicio 4.1: duración del viaje
A partir de `trips`, crea `viajes_dur` con una columna nueva `duracion_min` con los minutos entre `tpep_pickup_datetime` y `tpep_dropoff_datetime`, redondeados a 1 decimal.

Pista: `(F.unix_timestamp(fin) - F.unix_timestamp(inicio)) / 60`

In [0]:
viajes_dur = trips.withColumn(
    "duracion_min",
    F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 1)
)

display(viajes_dur.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duracion_min").limit(5))


tpep_pickup_datetime,tpep_dropoff_datetime,duracion_min
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,9.4
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,8.2
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,11.6
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,14.2
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,30.6


In [0]:
# comprobación
esperado = (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
assert "duracion_min" in viajes_dur.columns
assert viajes_dur.filter(F.abs(F.col("duracion_min") - esperado) > 0.06).count() == 0, "Revisa el cálculo de minutos"
ok()

Correcto


### Ejercicio 4.2: hora y día de la semana
A partir de `viajes_dur`, crea `viajes_tiempo` agregando dos columnas calculadas desde `tpep_pickup_datetime`:
- `hora`: hora del día (0 a 23)
- `dia_semana`: día de la semana como número (1 = domingo, 7 = sábado)

In [0]:
# viajes_tiempo = (
#     viajes_dur
#     .withColumn("hora", F.hour("tpep_pickup_datetime"))
#     .withColumn("dia_semana", F.dayofweek("tpep_pickup_datetime"))
# )

viajes_tiempo = (
    viajes_dur
    .select(
        "*",
        F.hour("tpep_pickup_datetime").alias("hora"),
        F.dayofweek("tpep_pickup_datetime").alias("dia_semana")
    )
)

display(viajes_tiempo.select("tpep_pickup_datetime", "hora", "dia_semana").limit(5))

tpep_pickup_datetime,hora,dia_semana
2016-02-13T21:47:53.000Z,21,7
2016-02-13T18:29:09.000Z,18,7
2016-02-06T19:40:58.000Z,19,7
2016-02-12T19:06:43.000Z,19,6
2016-02-23T10:27:56.000Z,10,3


In [0]:
# comprobación
r = viajes_tiempo.agg(F.min("hora"), F.max("hora"), F.min("dia_semana"), F.max("dia_semana")).first()
assert 0 <= r[0] and r[1] <= 23 and 1 <= r[2] and r[3] <= 7
assert viajes_tiempo.filter(F.col("hora") != F.hour("tpep_pickup_datetime")).count() == 0
ok()

Correcto


### Ejercicio 4.3: clasificar con `when`
A partir de `viajes_tiempo`, crea `viajes_tipo` con la columna `tipo_viaje`:
- `"corto"` si `trip_distance` < 2
- `"medio"` si `trip_distance` < 10
- `"largo"` en los demás casos

In [0]:
%sql
SELECT 
    *,
    CASE
        WHEN trip_distance < 2 THEN "corto"
        WHEN trip_distance < 10 THEN "medio"
        ELSE "largo"
    END AS tipo_viaje
FROM samples.nyctaxi.trips

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,tipo_viaje
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,corto
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,corto
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,corto
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,medio
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,medio
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069,corto
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003,largo
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201,largo
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023,medio
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222,medio


In [0]:
viajes_tipo = viajes_tiempo.withColumn(
    "tipo_viaje",
    F.when(F.col("trip_distance") < 2, "corto")
    .when(F.col("trip_distance") < 10, "medio")
    .otherwise("largo")
)

display(viajes_tipo.groupBy("tipo_viaje").count())

tipo_viaje,count
corto,12716
medio,8085
largo,1131


In [0]:
# comprobación
mal = viajes_tipo.filter(
    ((F.col("trip_distance") < 2) & (F.col("tipo_viaje") != "corto")) |
    ((F.col("trip_distance") >= 2) & (F.col("trip_distance") < 10) & (F.col("tipo_viaje") != "medio")) |
    ((F.col("trip_distance") >= 10) & (F.col("tipo_viaje") != "largo"))
).count()
assert mal == 0, f"{mal} filas mal clasificadas"
ok()

Correcto


### Ejercicio 4.4: división segura
A partir de `viajes_tipo`, crea `viajes` con la columna `tarifa_por_milla` = `fare_amount / trip_distance` redondeada a 2 decimales. Este va a ser el DataFrame principal de aquí en adelante.

Hay viajes con distancia 0, así que usa `F.try_divide` para que esos queden en `null` y no salte un error.

In [0]:
viajes = viajes_tipo.withColumn(
    "tarifa_por_milla",
    F.round(F.try_divide(F.col("fare_amount"), F.col("trip_distance")), 2)
)

display(viajes.select("trip_distance", "fare_amount", "tarifa_por_milla").limit(10))

trip_distance,fare_amount,tarifa_por_milla
1.4,8.0,5.71
1.31,7.5,5.73
1.8,9.5,5.28
2.3,11.5,5.0
2.6,18.5,7.12
1.4,6.5,4.64
10.4,31.0,2.98
10.15,28.5,2.81
3.27,15.0,4.59
4.42,15.0,3.39


In [0]:
display(viajes)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,duracion_min,hora,dia_semana,tipo_viaje,tarifa_por_milla
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,9.4,21,7,corto,5.71
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,8.2,18,7,corto,5.73
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,11.6,19,7,corto,5.28
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,14.2,19,6,medio,5.0
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,30.6,10,3,medio,7.12
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069,5.2,0,7,corto,4.64
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003,23.0,23,5,largo,2.98
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201,16.6,20,5,largo,2.81
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023,19.3,10,4,medio,4.59
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222,13.4,1,6,medio,3.39


In [0]:
# comprobación
assert viajes.filter((F.col("trip_distance") == 0) & F.col("tarifa_por_milla").isNotNull()).count() == 0
assert viajes.filter((F.col("trip_distance") > 0) & F.col("tarifa_por_milla").isNull()).count() == 0
ok("Correcto. viajes ya tiene duracion_min, hora, dia_semana, tipo_viaje y tarifa_por_milla")

Correcto. viajes ya tiene duracion_min, hora, dia_semana, tipo_viaje y tarifa_por_milla


Algo a notar: las celdas de 4.1 a 4.4 todavía no ejecutaron nada, solo describieron transformaciones (evaluación perezosa). Con `explain()` podemos ver el plan que Spark armó y optimizó para `viajes`:

In [0]:
viajes.explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [tpep_pickup_datetime#13355, tpep_dropoff_datetime#13356, trip_distance#13357, fare_amount#13358, pickup_zip#13359, dropoff_zip#13360, round((cast((unix_timestamp(tpep_dropoff_datetime#13356, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true) - unix_timestamp(tpep_pickup_datetime#13355, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true)) as double) / 60.0), 1) AS duracion_min#13364, hour(tpep_pickup_datetime#13355, Some(Etc/UTC)) AS hora#13409, dayofweek(cast(tpep_pickup_datetime#13355 as date)) AS dia_semana#13410, CASE WHEN (trip_distance#13357 < 2.0) THEN corto WHEN (trip_distance#13357 < 10.0) THEN medio ELSE largo END AS tipo_viaje#13555, round((fare_amount#13358 / trip_distance#13357), 2) AS tarifa_por_milla#13613]
      +- PhotonScan parquet samples.nyctaxi.trips[tpep_pickup_datetime#13355,tpep_dropoff_datetime#13356,trip_distance#13357,fare_amount#13358,pickup_zip#13359,dropoff_zip#13360] DataFilters: [], Dictionary

## Módulo 5. Agregaciones

`groupBy(...)` junto con `agg(...)` resume filas por grupo, igual que un `GROUP BY` en SQL.

```python
df.groupBy("columna").agg(
    F.count("*").alias("n"),
    F.avg("otra").alias("media"),
)
```
Las funciones de agregación más comunes son `count`, `sum`, `avg`, `min`, `max` y `countDistinct`. Si necesitas filtrar después de agregar (lo que en SQL sería `HAVING`), aplica `filter` sobre el resultado.

### Ejercicio 5.1: viajes por hora
Crea `por_hora` con el número de viajes por `hora`, ordenado por hora. La columna del conteo se tiene que llamar `n_viajes`.

Cuando salga el resultado del `display`, prueba el botón + > Visualización para armar un gráfico de barras.

In [0]:
display(viajes.limit(5))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,duracion_min,hora,dia_semana,tipo_viaje,tarifa_por_milla
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,9.4,21,7,corto,5.71
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,8.2,18,7,corto,5.73
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,11.6,19,7,corto,5.28
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,14.2,19,6,medio,5.0
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,30.6,10,3,medio,7.12


In [0]:
por_hora = (
    viajes
    .groupBy("hora")
    .agg(F.count("*").alias("n_viajes"))
    .orderBy("hora")
)
display(por_hora)

hora,n_viajes
0,745
1,573
2,445
3,308
4,250
5,220
6,475
7,801
8,1035
9,1044


In [0]:
# comprobación
assert por_hora.columns == ["hora", "n_viajes"]
assert por_hora.agg(F.sum("n_viajes")).first()[0] == viajes.count()
horas = [r["hora"] for r in por_hora.collect()]
assert horas == sorted(horas), "Ordena por hora"
ok()

Correcto


### Ejercicio 5.2: resumen por tipo de viaje
Crea `por_tipo` agrupando por `tipo_viaje`, con estas columnas:
- `n_viajes`: número de viajes
- `tarifa_media`: promedio de `fare_amount`, redondeado a 2 decimales
- `distancia_total`: suma de `trip_distance`, redondeada a 1 decimal

In [0]:
por_tipo = (
    viajes
    .groupBy("tipo_viaje")
    .agg(
        F.count("*").alias("n_viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_media"),
        F.round(F.sum("trip_distance"), 1).alias("distancia_total")
    )
)

display(por_tipo)

tipo_viaje,n_viajes,tarifa_media,distancia_total
corto,12716,7.19,13989.6
medio,8085,15.97,31827.7
largo,1131,44.47,16751.0


In [0]:
# comprobación
assert por_tipo.columns == ["tipo_viaje", "n_viajes", "tarifa_media", "distancia_total"]
assert por_tipo.count() == 3
assert por_tipo.agg(F.sum("n_viajes")).first()[0] == viajes.count()
ok()

Correcto


### Ejercicio 5.3: top zonas de origen
Crea `top_zips` con los 10 `pickup_zip` que tienen más viajes, considerando solo los que tienen al menos 50 viajes. Columnas: `pickup_zip`, `n_viajes` y `tarifa_media` (redondeada a 2). Ordena de más a menos viajes.

In [0]:
top_zips = (
    viajes
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("n_viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_media")
    )
    .filter(F.col("n_viajes") >= 50) # where es un alias de filter
    .orderBy(F.desc("n_viajes"))
    .limit(10)
)

display(top_zips)

pickup_zip,n_viajes,tarifa_media
10001,1227,10.62
10003,1181,10.98
10011,1129,10.91
10021,1021,10.21
10018,1012,11.4
10023,1008,10.04
10028,929,10.21
10012,834,11.35
10110,763,10.9
10065,702,9.81


In [0]:
# comprobación
filas = top_zips.collect()
assert top_zips.columns == ["pickup_zip", "n_viajes", "tarifa_media"]
assert 0 < len(filas) <= 10 and all(r["n_viajes"] >= 50 for r in filas)
assert all(filas[i]["n_viajes"] >= filas[i + 1]["n_viajes"] for i in range(len(filas) - 1))
ok()

Correcto


## Módulo 6. Joins

`df1.join(df2, condicion, how="tipo")` combina dos DataFrames. Los tipos que más vamos a usar:

| `how=` | Qué devuelve |
|---|---|
| `"inner"` (por defecto) | solo las filas que tienen pareja en ambos lados |
| `"left"` | todas las filas de la izquierda, con `null` donde no hay pareja |
| `"full"` | todas las filas de ambos lados |
| `"left_anti"` | las filas de la izquierda que no tienen pareja (muy útil para control de calidad) |

Si una de las tablas es pequeña, `F.broadcast(tabla)` la copia a cada executor y así se evita el shuffle.

Para los ejercicios vamos a usar esta tabla de códigos postales (simplificada, no están todos):

In [0]:
zonas = spark.createDataFrame(
    [
        (10001, "Chelsea", "Manhattan"), (10002, "Lower East Side", "Manhattan"),
        (10003, "East Village", "Manhattan"), (10011, "Chelsea", "Manhattan"),
        (10012, "SoHo", "Manhattan"), (10013, "Tribeca", "Manhattan"),
        (10014, "West Village", "Manhattan"), (10016, "Murray Hill", "Manhattan"),
        (10017, "Midtown East", "Manhattan"), (10019, "Midtown West", "Manhattan"),
        (10022, "Midtown East", "Manhattan"), (10023, "Upper West Side", "Manhattan"),
        (10036, "Times Square", "Manhattan"), (11201, "Brooklyn Heights", "Brooklyn"),
        (11211, "Williamsburg", "Brooklyn"), (11101, "Long Island City", "Queens"),
        (11371, "Aeropuerto LaGuardia", "Queens"), (11430, "Aeropuerto JFK", "Queens"),
    ],
    ["zip", "barrio", "distrito"],
)
display(zonas)

zip,barrio,distrito
10001,Chelsea,Manhattan
10002,Lower East Side,Manhattan
10003,East Village,Manhattan
10011,Chelsea,Manhattan
10012,SoHo,Manhattan
10013,Tribeca,Manhattan
10014,West Village,Manhattan
10016,Murray Hill,Manhattan
10017,Midtown East,Manhattan
10019,Midtown West,Manhattan


### Ejercicio 6.1: inner join
Crea `viajes_zona` uniendo `viajes` con `zonas` donde `pickup_zip == zip` (inner join).

In [0]:
viajes_zona = viajes.join(zonas, viajes["pickup_zip"] == zonas["zip"], how="inner")
print(viajes.count(), viajes_zona.count())


display(viajes_zona.select("pickup_zip", "barrio", "distrito", "fare_amount").limit(5))

21932 9344


pickup_zip,barrio,distrito,fare_amount
10023,Upper West Side,Manhattan,7.5
10001,Chelsea,Manhattan,9.5
10023,Upper West Side,Manhattan,6.5
11371,Aeropuerto LaGuardia,Queens,31.0
11371,Aeropuerto LaGuardia,Queens,28.5


In [0]:
# comprobación
assert "barrio" in viajes_zona.columns and "distrito" in viajes_zona.columns
assert viajes_zona.filter(F.col("barrio").isNull()).count() == 0, "En un inner join no debería haber barrios nulos"
ok()

Correcto


### Ejercicio 6.2: left join
Crea `viajes_zona_left` con un left join. Tiene que conservar todos los viajes.

In [0]:
viajes_zona_left = viajes.join(zonas, viajes["pickup_zip"] == zonas["zip"], how="left")
print(viajes.count(), viajes_zona_left.count())

21932 21932


In [0]:
display(viajes_zona_left)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,duracion_min,hora,dia_semana,tipo_viaje,tarifa_por_milla,zip,barrio,distrito
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,9.4,21,7,corto,5.71,null,null,null
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,8.2,18,7,corto,5.73,10023,Upper West Side,Manhattan
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,11.6,19,7,corto,5.28,10001,Chelsea,Manhattan
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,14.2,19,6,medio,5.0,null,null,null
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,30.6,10,3,medio,7.12,null,null,null
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069,5.2,0,7,corto,4.64,10023,Upper West Side,Manhattan
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003,23.0,23,5,largo,2.98,11371,Aeropuerto LaGuardia,Queens
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201,16.6,20,5,largo,2.81,11371,Aeropuerto LaGuardia,Queens
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023,19.3,10,4,medio,4.59,10014,West Village,Manhattan
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222,13.4,1,6,medio,3.39,10003,East Village,Manhattan


In [0]:
# comprobación
assert viajes_zona_left.count() == viajes.count()
ok()

Correcto


### Ejercicio 6.3: viajes sin zona
Crea `sin_zona` con los viajes cuyo `pickup_zip` no está en `zonas`. Usa `left_anti`.

In [0]:
sin_zona = viajes.join(zonas, viajes["pickup_zip"] == zonas["zip"], how="left_anti")
print(f"{sin_zona.count():,} viajes sin zona conocida")

12,588 viajes sin zona conocida


In [0]:
display(zonas)

zip,barrio,distrito
10001,Chelsea,Manhattan
10002,Lower East Side,Manhattan
10003,East Village,Manhattan
10011,Chelsea,Manhattan
10012,SoHo,Manhattan
10013,Tribeca,Manhattan
10014,West Village,Manhattan
10016,Murray Hill,Manhattan
10017,Midtown East,Manhattan
10019,Midtown West,Manhattan


In [0]:
display(sin_zona.select("pickup_zip").distinct())

pickup_zip
10452
11435
10026
10199
7737
11226
10039
11356
10024
10034


In [0]:
# comprobación
assert sin_zona.columns == viajes.columns, "left_anti solo devuelve las columnas de la izquierda"
assert sin_zona.count() + viajes_zona.count() == viajes.count()
ok()

Correcto


### Ejercicio 6.4: broadcast + agregación
Crea `por_distrito` con el número de viajes (`n_viajes`) y la tarifa media (`tarifa_media`, 2 decimales) por `distrito`. Haz el join usando `F.broadcast(zonas)` y ordena por `n_viajes` de mayor a menor.

In [0]:
por_distrito = (
    viajes
    .join(F.broadcast(zonas), viajes["pickup_zip"] == zonas["zip"], how="inner")
    .groupBy("distrito")
    .agg(
        F.count("*").alias("n_viajes"),
        F.round(F.avg("fare_amount"), 2).alias("tarifa_media")
    )
    .orderBy(F.desc("n_viajes"))
)

display(por_distrito)

distrito,n_viajes,tarifa_media
Manhattan,8688,10.98
Queens,507,30.18
Brooklyn,149,12.92


In [0]:
# comprobación
assert por_distrito.columns == ["distrito", "n_viajes", "tarifa_media"]
assert por_distrito.agg(F.sum("n_viajes")).first()[0] == viajes_zona.count()
ok()

Correcto


## Módulo 7. Funciones de ventana

Una ventana calcula un valor para cada fila mirando un grupo de filas relacionadas, pero sin colapsarlas en una sola como hace `groupBy`.

```python
w = Window.partitionBy("grupo").orderBy(F.desc("valor"))
df.withColumn("posicion", F.row_number().over(w))
```
Las más usadas son `row_number`, `rank`, `dense_rank`, `lag` y `lead`, y también agregaciones como `sum` sobre una ventana (para acumulados).

### Ejercicio 7.1: top 3 zonas por día de la semana
Crea `top3_por_dia` con los 3 `pickup_zip` con más viajes para cada `dia_semana`. Columnas: `dia_semana`, `pickup_zip`, `n_viajes`, `posicion`. El conteo por día y zip ya está hecho en `conteo`.

In [0]:
conteo = viajes.groupBy("dia_semana", "pickup_zip").agg(F.count("*").alias("n_viajes"))
# display(conteo)

w = Window.partitionBy("dia_semana").orderBy(F.desc("n_viajes"))

top3_por_dia = (
    conteo
    .withColumn("posicion", F.row_number().over(w))
    .where(F.col("posicion") <= 3)
)

display(top3_por_dia)

dia_semana,pickup_zip,n_viajes,posicion
1,10003,198,1
1,10018,169,2
1,10011,169,3
2,10001,179,1
2,10021,156,2
2,10028,148,3
3,10003,145,1
3,10021,142,2
3,10001,141,3
4,10001,184,1


In [0]:
# comprobación
assert top3_por_dia.columns == ["dia_semana", "pickup_zip", "n_viajes", "posicion"]
assert top3_por_dia.groupBy("dia_semana").count().agg(F.max("count")).first()[0] <= 3
ok()

Correcto


### Ejercicio 7.2: acumulado diario
Crea `por_fecha` con una fila por día y estas columnas:
- `fecha`: `F.to_date("tpep_pickup_datetime")`
- `n_viajes`: viajes de ese día
- `acumulado`: suma acumulada de `n_viajes` hasta ese día (incluido)

Las dos primeras ya están en `diario`, te falta el acumulado.

Pista: `Window.orderBy("fecha").rowsBetween(Window.unboundedPreceding, Window.currentRow)`

In [0]:
diario = (viajes
    .groupBy(F.to_date("tpep_pickup_datetime").alias("fecha"))
    .agg(F.count("*").alias("n_viajes"))
)

w = Window.orderBy("fecha").rowsBetween(Window.unboundedPreceding, Window.currentRow)
por_fecha = diario.withColumn("acumulado", F.sum("n_viajes").over(w))

# display(diario)
display(por_fecha)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fecha,n_viajes,acumulado
2016-01-01,326,326
2016-01-02,338,664
2016-01-03,276,940
2016-01-04,325,1265
2016-01-05,359,1624
2016-01-06,360,1984
2016-01-07,356,2340
2016-01-08,390,2730
2016-01-09,412,3142
2016-01-10,347,3489


In [0]:
# comprobación
ultima = por_fecha.orderBy(F.desc("fecha")).first()
assert ultima["acumulado"] == viajes.count(), "El último acumulado debe ser igual al total de viajes"
ok()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Correcto


### Ejercicio 7.3: variación respecto al día anterior
A partir de `por_fecha`, crea `variacion_diaria` con la columna `variacion`, que es `n_viajes` menos los `n_viajes` del día anterior. Usa `F.lag`.

In [0]:
# display(por_fecha)
w = Window.orderBy("fecha")
variacion_diaria = (
    por_fecha
    .withColumn("variacion", F.col("n_viajes") - F.lag("n_viajes", 1).over(w))
)

display(variacion_diaria)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fecha,n_viajes,acumulado,variacion
2016-01-01,326,326,null
2016-01-02,338,664,12
2016-01-03,276,940,-62
2016-01-04,325,1265,49
2016-01-05,359,1624,34
2016-01-06,360,1984,1
2016-01-07,356,2340,-4
2016-01-08,390,2730,34
2016-01-09,412,3142,22
2016-01-10,347,3489,-65


In [0]:
# comprobación
filas = variacion_diaria.orderBy("fecha").collect()
assert filas[0]["variacion"] is None, "El primer día no tiene día anterior: debería ser null"
assert all(filas[i]["variacion"] == filas[i]["n_viajes"] - filas[i - 1]["n_viajes"] for i in range(1, len(filas)))
ok()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Correcto


## Módulo 8. Spark SQL

Sobre los mismos datos se puede mezclar SQL y Python:

1. Registras el DataFrame como vista temporal con `df.createOrReplaceTempView("nombre")`
2. Consultas con `spark.sql("SELECT ...")`, que devuelve un DataFrame
3. O usas una celda que empiece con `%sql`

Las vistas temporales solo existen en tu sesión. En serverless no hay vistas temporales globales.

### Ejercicio 8.1: de DataFrame a SQL
Registra `viajes` como la vista temporal `viajes` y después, con `spark.sql`, crea `res_sql` con el número de viajes (`n_viajes`) por `tipo_viaje`.

In [0]:
viajes.createOrReplaceTempView("viajes")
res_sql = spark.sql(
    """
    SELECT tipo_viaje, count(1) AS n_viajes
    FROM viajes
    GROUP BY tipo_viaje
    """
)

display(res_sql)

tipo_viaje,n_viajes
corto,12716
medio,8085
largo,1131


In [0]:
%sql
SELECT *
FROM viajes

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,duracion_min,hora,dia_semana,tipo_viaje,tarifa_por_milla
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110,9.4,21,7,corto,5.71
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023,8.2,18,7,corto,5.73
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018,11.6,19,7,corto,5.28
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111,14.2,19,6,medio,5.0
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022,30.6,10,3,medio,7.12
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069,5.2,0,7,corto,4.64
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003,23.0,23,5,largo,2.98
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201,16.6,20,5,largo,2.81
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023,19.3,10,4,medio,4.59
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222,13.4,1,6,medio,3.39


In [0]:
# comprobación
a = {r["tipo_viaje"]: r["n_viajes"] for r in res_sql.collect()}
b = {r["tipo_viaje"]: r["n_viajes"] for r in por_tipo.collect()}
assert a == b, "El resultado en SQL debería coincidir con por_tipo del ejercicio 5.2"
ok()

Correcto


### Ejercicio 8.2: celda `%sql`
En la celda de abajo escribe una consulta SQL directamente sobre `samples.nyctaxi.trips` que devuelva los 5 `pickup_zip` con mayor tarifa promedio (`tarifa_media`, redondeada a 2), tomando solo los zips con al menos 20 viajes.

Pista: `GROUP BY` + `HAVING COUNT(*) >= 20` + `ORDER BY ... DESC LIMIT 5`

In [0]:
%sql
SELECT
    pickup_zip,
    ROUND(AVG(fare_amount), 2) AS tarifa_media,
    COUNT(1) AS n_viajes
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
HAVING n_viajes >= 20
ORDER BY tarifa_media DESC
LIMIT 5


pickup_zip,tarifa_media,n_viajes
11422,44.78,429
11371,30.64,483
11370,25.31,26
11109,17.8,23
10280,16.33,103


### Ejercicio 8.3: SQL y DataFrames juntos
Con `spark.sql`, saca los viajes en hora punta de la mañana (`hora` entre 7 y 9, inclusive) desde la vista `viajes`. Luego, con la API de Python, agrupa por `tipo_viaje` contando los viajes (`n_viajes`). Guarda el resultado en `hora_punta`.

In [0]:
hora_punta = (
    spark.sql("SELECT * FROM viajes WHERE hora BETWEEN 7 AND 9")
    .groupBy("tipo_viaje")
    .agg(F.count("*").alias("n_viajes"))
)

display(hora_punta)

tipo_viaje,n_viajes
corto,1799
medio,963
largo,118


In [0]:
# comprobación
assert hora_punta.agg(F.sum("n_viajes")).first()[0] == viajes.filter(F.col("hora").between(7, 9)).count()
ok()

Correcto


## Módulo 9. Limpieza de datos

Para datos sucios tenemos:

| Problema | Qué usar |
|---|---|
| filas duplicadas | `dropDuplicates()` o `dropDuplicates(["col"])` |
| nulos | `fillna({"col": valor})`, `dropna(subset=[...])`, `F.col("c").isNull()` |
| espacios y mayúsculas | `F.trim`, `F.lower`, `F.upper`, `F.initcap` |
| texto que debería ser número | `F.expr("try_cast(col AS INT)")`, devuelve `null` si no se puede convertir |

Con ANSI activo, `F.col("edad").cast("int")` sobre un `"N/A"` lanza error, mientras que `try_cast` devuelve `null`.

In [0]:
clientes_sucios = spark.createDataFrame(
    [
        (1, "  ana pérez ", "ANA@MAIL.COM", "34", "lima"),
        (2, "Luis Gómez", "luis@mail.com", None, "Bogotá"),
        (2, "Luis Gómez", "luis@mail.com", None, "Bogotá"),
        (3, "MARTA RUIZ", None, "29", "quito"),
        (4, "jorge díaz", "jorge@mail.com", "N/A", "Lima"),
        (5, None, "sofia@mail.com", "41", None),
    ],
    "id INT, nombre STRING, email STRING, edad STRING, ciudad STRING",
)
display(clientes_sucios)

id,nombre,email,edad,ciudad
1,ana pérez,ANA@MAIL.COM,34,lima
2,Luis Gómez,luis@mail.com,null,Bogotá
2,Luis Gómez,luis@mail.com,null,Bogotá
3,MARTA RUIZ,null,29,quito
4,jorge díaz,jorge@mail.com,N/A,Lima
5,null,sofia@mail.com,41,null


### Ejercicio 9.1: nulos por columna
Crea `nulos`, un DataFrame de una sola fila con la cantidad de valores nulos de cada columna de `clientes_sucios`.

Pista: `F.col(c).isNull().cast("int")` vale 1 cuando el valor es nulo. Súmalo con `F.sum` para cada columna `c` usando una list comprehension, y ponle `alias(c)` para que cada columna conserve su nombre.

In [0]:
nulos = clientes_sucios.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in clientes_sucios.columns
    ]
)

display(nulos)

id,nombre,email,edad,ciudad
0,1,1,2,1


In [0]:
# comprobación
fila = nulos.first()
assert nulos.columns == clientes_sucios.columns
assert (fila["id"], fila["nombre"], fila["email"], fila["edad"], fila["ciudad"]) == (0, 1, 1, 2, 1)
ok()

Correcto


### Ejercicio 9.2: quitar duplicados
Crea `clientes_unicos` sin las filas duplicadas.

In [0]:
clientes_unicos = clientes_sucios.dropDuplicates()
print(clientes_unicos.count())

5


In [0]:
# comprobación
assert clientes_unicos.count() == 5
ok()

Correcto


### Ejercicio 9.3: normalizar y rellenar
A partir de `clientes_unicos`, crea `clientes_limpios` así:
- `nombre`: sin espacios de sobra y con mayúscula inicial en cada palabra (`"  ana pérez "` pasa a `"Ana Pérez"`)
- `email`: en minúsculas
- `edad`: convertida a entero con `try_cast` (`"N/A"` queda en `null`)
- `ciudad`: con mayúscula inicial
- nulos: `nombre` se rellena con `"Desconocido"` y `ciudad` con `"Sin ciudad"`

In [0]:
clientes_limpios = (
    clientes_unicos
    .select(
        "id",
        F.initcap(F.trim("nombre")).alias("nombre"),
        F.lower("email").alias("email"),
        F.expr("try_cast(edad AS INT)").alias("edad"),
        F.initcap("ciudad").alias("ciudad")
    )
    .fillna({
        "nombre": "Desconocido",
        "ciudad": "Sin ciudad"
    })
)

display(clientes_limpios.orderBy("id"))

id,nombre,email,edad,ciudad
1,Ana Pérez,ana@mail.com,34,Lima
2,Luis Gómez,luis@mail.com,null,Bogotá
3,Marta Ruiz,null,29,Quito
4,Jorge Díaz,jorge@mail.com,null,Lima
5,Desconocido,sofia@mail.com,41,Sin ciudad


In [0]:
# comprobación
c = {r["id"]: r for r in clientes_limpios.collect()}
assert dict(clientes_limpios.dtypes)["edad"] == "int", "edad debe ser entero"
assert c[1]["nombre"] == "Ana Pérez" and c[1]["email"] == "ana@mail.com" and c[1]["ciudad"] == "Lima"
assert c[4]["edad"] is None and c[3]["edad"] == 29
assert c[5]["nombre"] == "Desconocido" and c[5]["ciudad"] == "Sin ciudad"
ok()

Correcto


### Ejercicio 9.4: descartar registros inválidos
A partir de `clientes_limpios`, crea `clientes_validos` quitando los clientes que no tienen email.

In [0]:
clientes_validos = clientes_limpios.dropna(subset=["email"])
display(clientes_validos)

id,nombre,email,edad,ciudad
1,Ana Pérez,ana@mail.com,34,Lima
2,Luis Gómez,luis@mail.com,null,Bogotá
4,Jorge Díaz,jorge@mail.com,null,Lima
5,Desconocido,sofia@mail.com,41,Sin ciudad


In [0]:
# comprobación
assert clientes_validos.count() == 4 and clientes_validos.filter(F.col("email").isNull()).count() == 0
ok()

Correcto


## Módulo 10. Escribir tablas y archivos

En Databricks normalmente escribimos en dos lugares:

- Tablas de Unity Catalog (en formato Delta por defecto): `df.write.mode("overwrite").saveAsTable("catalogo.esquema.tabla")`
- Archivos dentro de un volumen, con rutas tipo `/Volumes/catalogo/esquema/volumen/carpeta`

Modos de escritura: `"overwrite"`, `"append"`, `"ignore"` y `"error"` (el de por defecto, falla si la tabla ya existe).

### Ejercicio 10.1: guardar como tabla Delta
Guarda `por_tipo` (del ejercicio 5.2) como la tabla `workspace.default.taxis_por_tipo`, sobrescribiendo si ya existe. Después léela de nuevo en `tabla_leida`.

In [0]:
nombre_tabla = f"{CATALOGO}.{ESQUEMA}.taxis_por_tipo"
por_tipo.write.mode("overwrite").saveAsTable(nombre_tabla)

tabla_leida = spark.sql(f"SELECT * FROM {nombre_tabla}")

display(tabla_leida)

tipo_viaje,n_viajes,tarifa_media,distancia_total
corto,12716,7.19,13989.6
largo,1131,44.47,16751.0
medio,8085,15.97,31827.7


In [0]:
tabla_leida = spark.read.table(nombre_tabla)
display(tabla_leida)

tipo_viaje,n_viajes,tarifa_media,distancia_total
corto,12716,7.19,13989.6
largo,1131,44.47,16751.0
medio,8085,15.97,31827.7


In [0]:
# comprobación
assert tabla_leida.count() == por_tipo.count() == 3
ok("Correcto. La tabla aparece en Catalog > workspace > default")

Correcto. La tabla aparece en Catalog > workspace > default


Delta guarda un historial de cada escritura. Ejecuta esta celda para verlo; si vuelves a correr el 10.1 vas a ver una versión nueva.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.taxis_por_tipo"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-04T01:20:24.000Z,76075192950026,codelab.genius.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(316195817464850),accd0ddc-e14e-45ec-b77e-cc34eb569a55,1104-003910-24tciyyr-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1615)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


### Ejercicio 10.2: CSV en un volumen
Primero creamos un volumen, que es una carpeta para archivos gobernada por Unity Catalog:

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.curso_pyspark")
RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/curso_pyspark"
print(RUTA_VOLUMEN)

/Volumes/workspace/default/curso_pyspark


Ahora:
1. Escribe `top_zips` (ejercicio 5.3) como CSV con encabezado en `RUTA_VOLUMEN + "/top_zips"`, en modo overwrite.
2. Léelo de vuelta en `top_zips_leido`, con encabezado e inferencia de tipos.

In [0]:
# display(top_zips)

In [0]:
ruta_csv = RUTA_VOLUMEN + "/top_zips"
(
    top_zips
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(ruta_csv)
)

top_zips_leido = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_csv)
)

display(top_zips_leido)

pickup_zip,n_viajes,tarifa_media
10001,1227,10.62
10003,1181,10.98
10011,1129,10.91
10021,1021,10.21
10018,1012,11.4
10023,1008,10.04
10028,929,10.21
10012,834,11.35
10110,763,10.9
10065,702,9.81


In [0]:
# comprobación
assert top_zips_leido.count() == top_zips.count()
assert set(top_zips_leido.columns) == {"pickup_zip", "n_viajes", "tarifa_media"}
ok()

Correcto


Fíjate que Spark no escribe un único archivo sino una carpeta con uno o varios archivos `part-*` (uno por partición):

In [0]:
display(dbutils.fs.ls(ruta_csv))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/curso_pyspark/top_zips/_SUCCESS,_SUCCESS,0,1793755530000
dbfs:/Volumes/workspace/default/curso_pyspark/top_zips/_committed_7372174668508944460,_committed_7372174668508944460,113,1793755530000
dbfs:/Volumes/workspace/default/curso_pyspark/top_zips/_started_7372174668508944460,_started_7372174668508944460,0,1793755530000
dbfs:/Volumes/workspace/default/curso_pyspark/top_zips/part-00000-tid-7372174668508944460-2cc33896-dc38-4677-90cb-80d9971f050d-394-1-c000.csv,part-00000-tid-7372174668508944460-2cc33896-dc38-4677-90cb-80d9971f050d-394-1-c000.csv,196,1793755530000


## Desafío final: ventas por país con TPC-H

Aquí toca juntar todo lo anterior. Las tablas de `samples.tpch` simulan un sistema de pedidos:

| Tabla | Columnas que vas a necesitar |
|---|---|
| `orders` | `o_orderkey`, `o_custkey`, `o_totalprice`, `o_orderdate` |
| `customer` | `c_custkey`, `c_nationkey` |
| `nation` | `n_nationkey`, `n_name`, `n_regionkey` |
| `region` | `r_regionkey`, `r_name` |

In [0]:
orders   = spark.read.table("samples.tpch.orders")
customer = spark.read.table("samples.tpch.customer")
nation   = spark.read.table("samples.tpch.nation")
region   = spark.read.table("samples.tpch.region")
display(orders.limit(5))

o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment
11396166,179329,O,187683.10,1996-06-22,4-NOT SPECIFIED,Clerk#000004689,0,"ep fluffily regular packages. regular, final courts ag"
11396167,473245,F,117554.58,1994-07-20,1-URGENT,Clerk#000001856,0,efully after the carefully express packages; final r
11396192,77549,O,47611.56,1996-01-03,3-MEDIUM,Clerk#000001450,0,es. regular tithes poach careful
11396193,610651,F,121869.26,1994-11-29,5-LOW,Clerk#000003696,0,"are furiously along the bold, even ideas. even, final pinto beans"
11396194,193169,O,241066.47,1998-03-21,4-NOT SPECIFIED,Clerk#000002476,0,lar instructions use slyly carefully furious instructions. quickly


In [0]:
display(customer.limit(5))

c_custkey,c_name,c_address,c_nationkey,c_phone,c_acctbal,c_mktsegment,c_comment
412445,Customer#000412445,"0QAB3OjYnbP6mA0B,kgf",21,31-421-403-4333,5358.33,BUILDING,arefully blithely regular epi
412446,Customer#000412446,"5u8MSbyiC7J,7PuY4Ivaq1JRbTCMKeNVqg",20,30-487-949-7942,9441.59,MACHINERY,"sleep according to the fluffily even forges. fluffily careful packages after the ironic, silent deposi"
412447,Customer#000412447,HC4ZT62gKPgrjr ceoaZgFOunlUogr7GO,7,17-797-466-6308,7868.75,AUTOMOBILE,aggle blithely among the carefully express excus
412448,Customer#000412448,hJok1MMrDgH,6,16-541-510-4964,6060.98,MACHINERY,ly silent requests boost slyly. express courts sleep according to the fluf
412449,Customer#000412449,"zAt1nZNG01gOhIqgyDtDa S,Y0VSofZJs1dd",14,24-710-983-5536,4973.84,HOUSEHOLD,"refully final theodolites. final, slow excuses sleep quickly! quickly ironic idea"


In [0]:
display(nation.limit(5))

n_nationkey,n_name,n_regionkey,n_comment
0,ALGERIA,0,haggle. carefully final deposits detect slyly agai
1,ARGENTINA,1,al foxes promise slyly according to the regular accounts. bold requests alon
2,BRAZIL,1,y alongside of the pending deposits. carefully special packages are about the ironic forges. slyly special
3,CANADA,1,"eas hang ironic, silent packages. slyly regular packages are furiously over the tithes. fluffily bold"
4,EGYPT,4,y above the carefully unusual theodolites. final dugouts are quickly across the furiously regular d


In [0]:
display(region.limit(5))

r_regionkey,r_name,r_comment
0,AFRICA,lar deposits. blithely final packages cajole. regular waters are final requests. regular accounts are according to
1,AMERICA,"hs use ironic, even requests. s"
2,ASIA,ges. thinly even pinto beans ca
3,EUROPE,ly final courts cajole furiously final excuse
4,MIDDLE EAST,uickly special accounts cajole carefully blithely close requests. carefully final asymptotes haggle furiousl


### Desafío A: top 5 países por facturación en 1995
Crea `top_paises` con las columnas `pais`, `n_pedidos` y `facturacion` (suma de `o_totalprice`, redondeada a 2), solo con los pedidos cuyo `o_orderdate` sea de 1995. Ordena por facturación de mayor a menor y quédate con 5 filas.

Un orden posible: filtrar `orders` por año, join con `customer`, join con `nation`, agrupar, ordenar y limitar.

In [0]:
top_paises = (
    orders
    .where(F.year("o_orderdate") == 1995)
    .join(customer, orders["o_custkey"] == customer["c_custkey"], how="inner")
    .join(nation, customer["c_nationkey"] == nation["n_nationkey"], how="inner")
    .groupBy(F.col("n_name").alias("pais"))
    .agg(
        F.count("*").alias("n_pedidos"),
        F.round(F.sum("o_totalprice"), 2).alias("facturacion")
    )
    .orderBy(F.desc("facturacion"))
    .limit(5)
)

display(top_paises)

pais,n_pedidos,facturacion
FRANCE,45818,6963662248.61
UNITED KINGDOM,46138,6959303330.34
IRAN,46050,6944920378.27
GERMANY,45696,6928140675.92
ARGENTINA,46010,6924104098.33


In [0]:
# comprobación
filas = top_paises.collect()
assert top_paises.columns == ["pais", "n_pedidos", "facturacion"] and len(filas) == 5
assert all(filas[i]["facturacion"] >= filas[i + 1]["facturacion"] for i in range(4))
ok("Desafío A listo")

Desafío A listo


### Desafío B (bonus): el mejor país de cada región
Considerando todos los años, crea `mejor_pais_region` con el país de mayor facturación total en cada región (`r_name`, que debe quedar como columna `region`). Columnas: `region`, `pais`, `facturacion`.

Idea: agrega por región y país, y después usa una ventana con `row_number` particionada por región.

In [0]:
fact_pais = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"], how="inner")
    .join(nation, customer["c_nationkey"] == nation["n_nationkey"], how="inner")
    .join(region, customer["n_regionkey"] == nation["r_regionkey"], how="inner")
    .groupby(F.col("r_name").alias("region"), F.col("n_name").alias("pais"))
    .agg(F.round(F.sum("o_totalprice"), 2).alias("facturacion"))
)

w = Window.partitionBy("region").orderBy(F.desc("facturacion"))

mejor_pais_region = (
    fact_pais
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .select("region", "pais", "facturacion")
    .orderBy("region")
)


display(mejor_pais_region)

region,pais,facturacion
AFRICA,ETHIOPIA,45602005018.40
AMERICA,ARGENTINA,45540967378.41
ASIA,INDONESIA,45921526909.21
EUROPE,GERMANY,45725194459.72
MIDDLE EAST,IRAQ,45968155784.04


In [0]:
# comprobación
assert mejor_pais_region.columns == ["region", "pais", "facturacion"]
assert mejor_pais_region.count() == region.count()
ok("Desafío B listo")

Desafío B listo


## Limpieza (opcional)
Si quieres borrar lo que creaste, descomenta estas líneas y ejecútalas:

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{ESQUEMA}.taxis_por_tipo")
spark.sql(f"DROP VOLUME IF EXISTS {CATALOGO}.{ESQUEMA}.curso_pyspark")

DataFrame[]

## Para seguir
- Documentación de PySpark: https://spark.apache.org/docs/latest/api/python/
- Databricks Free Edition: https://docs.databricks.com/aws/en/getting-started/free-edition
- Databricks Academy tiene cursos gratuitos para seguir practicando.
- Lo que viene después: Structured Streaming, MLlib y pipelines declarativos (Lakeflow), que usan la misma API de DataFrames.